# Notebook 06 — SQL Analysis

**Project:** APAC Territory Planning  
**Objective:** Replicate key metrics from notebooks 02-04 using pure SQL — demonstrating that all analysis is reproducible directly from the data without pandas transformations.

**Business Questions:**
1. Quota attainment by rep — SQL only
2. Pipeline coverage and win rate by subregion
3. Whitespace accounts by subregion and tier
4. Territory balance — accounts and ARR per rep

In [1]:
import pandas as pd
import duckdb
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

In [2]:
accounts      = pd.read_csv(f"{DATA_DIR}/accounts.csv")
reps          = pd.read_csv(f"{DATA_DIR}/reps.csv")
assignments   = pd.read_csv(f"{DATA_DIR}/assignments.csv")
opportunities = pd.read_csv(f"{DATA_DIR}/opportunities.csv")
customers     = pd.read_csv(f"{DATA_DIR}/customers.csv")
ws_scored     = pd.read_csv(f"{PROCESSED_DIR}/whitespace_scored.csv")

# Active customers only — consistent with notebooks 02-05
customers = customers[
    (pd.to_datetime(customers["contract_end"]) > pd.Timestamp("2025-03-01")) |
    (customers["renewal_flag"] == 1)
].copy()

con = duckdb.connect()
con.register("accounts",      accounts)
con.register("reps",          reps)
con.register("assignments",   assignments)
con.register("opportunities", opportunities)
con.register("customers",     customers)
con.register("ws_scored",     ws_scored)

print(f"Active customers loaded: {len(customers):,}")
print()
print(con.execute("SHOW TABLES").df().to_string(index=False))

Active customers loaded: 281

         name
     accounts
  assignments
    customers
opportunities
         reps
    ws_scored


## 1. APAC Territory Health Scorecard — Single SQL Query

A single consolidated query joining all 5 tables to produce a territory health 
scorecard combining coverage, performance, pipeline and whitespace metrics.

In [6]:
sql = """
WITH active_customers AS (
    SELECT *
    FROM customers
    WHERE contract_end > '2025-03-01'
       OR renewal_flag = 1
),
rep_quota AS (
    SELECT
        r.rep_id,
        r.rep_name,
        r.subregion,
        r.segment_focus,
        r.quota_usd,
        r.max_accounts
    FROM reps r
    WHERE r.subregion != 'Regional'
),
rep_accounts AS (
    SELECT
        r.rep_id,
        COUNT(DISTINCT asgn.account_id)     AS assigned_accounts,
        SUM(a.estimated_arr)                AS territory_arr
    FROM rep_quota r
    LEFT JOIN assignments asgn ON r.rep_id = asgn.rep_id
        AND asgn.coverage_status = 'Assigned'
    LEFT JOIN accounts a ON asgn.account_id = a.account_id
    GROUP BY r.rep_id
),
rep_revenue AS (
    SELECT
        r.rep_id,
        COALESCE(SUM(c.arr), 0)             AS actual_arr,
        COUNT(DISTINCT c.customer_id)        AS n_customers
    FROM rep_quota r
    LEFT JOIN active_customers c ON r.rep_id = c.rep_id
    GROUP BY r.rep_id
),
rep_pipeline AS (
    SELECT
        r.rep_id,
        COUNT(DISTINCT o.opportunity_id)
            FILTER (WHERE o.stage NOT IN ('Closed Won','Closed Lost'))
                                            AS open_opps,
        COALESCE(SUM(o.arr_potential)
            FILTER (WHERE o.stage NOT IN ('Closed Won','Closed Lost')), 0)
                                            AS open_pipeline,
        COUNT(DISTINCT o.opportunity_id)
            FILTER (WHERE o.stage = 'Closed Won')
                                            AS closed_won,
        COUNT(DISTINCT o.opportunity_id)
            FILTER (WHERE o.stage = 'Closed Lost')
                                            AS closed_lost
    FROM rep_quota r
    LEFT JOIN opportunities o ON r.rep_id = o.rep_id
    GROUP BY r.rep_id
)
SELECT
    q.rep_name,
    q.subregion,
    q.segment_focus                                                     AS segment,
    q.quota_usd                                                         AS quota,
    ra.assigned_accounts,
    q.max_accounts,
    ROUND(ra.assigned_accounts * 100.0 / q.max_accounts, 1)            AS load_pct,
    ROUND(ra.territory_arr, 0)                                          AS territory_arr,
    COALESCE(rr.actual_arr, 0)                                          AS actual_arr,
    ROUND(COALESCE(rr.actual_arr, 0) / q.quota_usd * 100, 1)           AS attainment_pct,
    rr.n_customers,
    rp.open_opps,
    ROUND(rp.open_pipeline, 0)                                          AS open_pipeline,
    ROUND(rp.open_pipeline / NULLIF(q.quota_usd, 0) * 100, 1)          AS pipeline_coverage_pct,
    ROUND(
        rp.closed_won * 100.0 / NULLIF(rp.closed_won + rp.closed_lost, 0)
    , 1)                                                                AS win_rate_pct
FROM rep_quota q
LEFT JOIN rep_accounts  ra ON q.rep_id = ra.rep_id
LEFT JOIN rep_revenue   rr ON q.rep_id = rr.rep_id
LEFT JOIN rep_pipeline  rp ON q.rep_id = rp.rep_id
ORDER BY attainment_pct DESC NULLS LAST
"""

with open("../sql/territory_health_scorecard.sql", "w") as f:
    f.write(sql)

scorecard = con.execute(sql).df()

print("APAC TERRITORY HEALTH SCORECARD")
print(f"{'Rep':<26} {'Subregion':<16} {'Segment':<12} {'Load%':>7} {'Attain%':>9} {'Win%':>7} {'Pipeline Cov%':>14}")
print("─" * 96)
for _, row in scorecard.iterrows():
    win_rate = f"{row['win_rate_pct']:.1f}%" if pd.notna(row['win_rate_pct']) else "N/A"
    print(f"{row['rep_name']:<26} {row['subregion']:<16} {row['segment']:<12} "
          f"{row['load_pct']:>6.1f}% {row['attainment_pct']:>8.1f}% "
          f"{win_rate:>6} {row['pipeline_coverage_pct']:>13.1f}%")

APAC TERRITORY HEALTH SCORECARD
Rep                        Subregion        Segment        Load%   Attain%    Win%  Pipeline Cov%
────────────────────────────────────────────────────────────────────────────────────────────────
Sara Ross                  SEA              SMB            88.0%    294.3%  22.2%        1019.5%
Jacob Wilson               ANZ              Mid-Market     92.5%    211.5%   0.0%         242.6%
Michael Roberts            SEA              SMB            90.7%    143.0%  42.9%         874.4%
Arthur Miller              Greater China    SMB            88.0%    129.5%  31.8%         826.5%
Anthony Mccarty            Greater China    SMB            86.7%    121.8%  37.5%        1018.7%
Lauren Butler              SEA              SMB            88.0%    105.7%  23.1%         899.0%
Timothy Lawson             North Asia       Mid-Market     92.5%     73.6%  50.0%         132.4%
Teresa Robbins             ANZ              Enterprise     87.5%     34.5%   0.0%          66.

## 2. Subregion Summary

In [7]:
sql = """
WITH active_customers AS (
    SELECT * FROM customers
    WHERE contract_end > '2025-03-01' OR renewal_flag = 1
),
subregion_metrics AS (
    SELECT
        a.subregion,
        COUNT(DISTINCT a.account_id)                                        AS total_accounts,
        COUNT(DISTINCT CASE WHEN asgn.coverage_status = 'Assigned' 
            THEN a.account_id END)                                          AS assigned_accounts,
        COUNT(DISTINCT CASE WHEN asgn.coverage_status = 'Unassigned' 
            THEN a.account_id END)                                          AS unassigned_accounts,
        COUNT(DISTINCT CASE WHEN a.is_customer = 1 
            THEN a.account_id END)                                          AS n_customers,
        COALESCE(SUM(CASE WHEN a.is_customer = 1 
            THEN c.arr END), 0)                                             AS customer_arr,
        COUNT(DISTINCT r.rep_id)                                            AS n_reps,
        SUM(DISTINCT r.quota_usd)                                           AS total_quota,
        COALESCE(SUM(CASE WHEN o.stage NOT IN ('Closed Won','Closed Lost')
            THEN o.arr_potential END), 0)                                   AS open_pipeline,
        COUNT(DISTINCT CASE WHEN o.stage = 'Closed Won' 
            THEN o.opportunity_id END)                                      AS closed_won,
        COUNT(DISTINCT CASE WHEN o.stage = 'Closed Lost' 
            THEN o.opportunity_id END)                                      AS closed_lost
    FROM accounts a
    LEFT JOIN assignments asgn ON a.account_id = asgn.account_id
    LEFT JOIN active_customers c ON a.account_id = c.account_id
    LEFT JOIN reps r ON asgn.rep_id = r.rep_id AND r.subregion != 'Regional'
    LEFT JOIN opportunities o ON a.account_id = o.account_id
    GROUP BY a.subregion
)
SELECT
    subregion,
    total_accounts,
    assigned_accounts,
    unassigned_accounts,
    ROUND(unassigned_accounts * 100.0 / total_accounts, 1)              AS whitespace_pct,
    n_customers,
    ROUND(n_customers * 100.0 / total_accounts, 1)                      AS customer_rate_pct,
    ROUND(customer_arr, 0)                                               AS customer_arr,
    n_reps,
    ROUND(open_pipeline, 0)                                              AS open_pipeline,
    ROUND(closed_won * 100.0 / NULLIF(closed_won + closed_lost, 0), 1)  AS win_rate_pct
FROM subregion_metrics
ORDER BY customer_arr DESC NULLS LAST
"""

with open("../sql/subregion_summary.sql", "w") as f:
    f.write(sql)

subregion = con.execute(sql).df()

print("SUBREGION SUMMARY")
print(f"{'Subregion':<16} {'Accounts':>9} {'Assigned':>9} {'WS%':>6} {'Customers':>10} {'Cust%':>7} {'Customer ARR':>13} {'Reps':>5} {'Win%':>7}")
print("─" * 90)
for _, row in subregion.iterrows():
    win_rate = f"{row['win_rate_pct']:.1f}%" if pd.notna(row['win_rate_pct']) else "N/A"
    print(f"{row['subregion']:<16} {row['total_accounts']:>9,} {row['assigned_accounts']:>9,} "
          f"{row['whitespace_pct']:>5.1f}% {row['n_customers']:>10,} "
          f"{row['customer_rate_pct']:>6.1f}% {row['customer_arr']:>13,.0f} "
          f"{row['n_reps']:>5,.0f} {win_rate:>7}")

SUBREGION SUMMARY
Subregion         Accounts  Assigned    WS%  Customers   Cust%  Customer ARR  Reps    Win%
──────────────────────────────────────────────────────────────────────────────────────────
SEA                    600       549   8.5%        133   22.2%     2,886,700     6   23.1%
ANZ                    200       109  45.5%         61   30.5%     2,742,600     2    0.0%
North Asia             400       220  45.0%        122   30.5%     2,590,800     3   22.2%
Greater China          480       411  14.4%         63   13.1%     1,573,700     4   32.7%
India                  320       263  17.8%         33   10.3%       152,500     3   36.4%


## 3. Whitespace Priority Summary

In [9]:
sql = """
SELECT
    subregion,
    tier,
    priority_flag,
    COUNT(account_id)           AS n_accounts,
    ROUND(SUM(estimated_arr),0) AS total_arr,
    ROUND(AVG(estimated_arr),0) AS avg_arr
FROM ws_scored
GROUP BY subregion, tier, priority_flag
ORDER BY subregion, tier, priority_flag
"""

with open("../sql/whitespace_priority_summary.sql", "w") as f:
    f.write(sql)

ws_summary = con.execute(sql).df()

print("WHITESPACE PRIORITY SUMMARY")
print(f"{'Subregion':<16} {'Tier':<8} {'Priority':<10} {'Accounts':>9} {'Total ARR':>14} {'Avg ARR':>10}")
print("─" * 72)
for _, row in ws_summary.iterrows():
    print(f"{row['subregion']:<16} {row['tier']:<8} {row['priority_flag']:<10} "
          f"{row['n_accounts']:>9,} {row['total_arr']:>14,.0f} {row['avg_arr']:>10,.0f}")

WHITESPACE PRIORITY SUMMARY
Subregion        Tier     Priority    Accounts      Total ARR    Avg ARR
────────────────────────────────────────────────────────────────────────
ANZ              Tier 1   Priority          38     53,772,700  1,415,071
ANZ              Tier 1   Standard          18      7,735,400    429,744
ANZ              Tier 2   Standard          62      9,405,800    151,706
ANZ              Tier 3   Standard          14        384,300     27,450
Greater China    Tier 1   Priority         105    136,491,000  1,299,914
Greater China    Tier 1   Standard          19      9,806,600    516,137
Greater China    Tier 2   Standard         152     22,075,700    145,235
Greater China    Tier 3   Standard          87      2,245,200     25,807
India            Tier 1   Priority          20     27,958,100  1,397,905
India            Tier 1   Standard           2        673,900    336,950
India            Tier 2   Standard          98     11,250,800    114,804
India            Tier 3

## Key Findings

1. **Single query scorecard:** All key territory metrics — load, attainment, win rate, pipeline coverage — produced in one SQL query joining 5 tables via chained CTEs
2. **SMB vs Enterprise attainment gap confirmed in SQL:** SMB reps 106-294% attainment, Enterprise reps 13-35% — territory design imbalance visible without any pandas transformation
3. **India weakest subregion by every metric:** $152K customer ARR, 10.3% customer rate, 6-16% rep attainment — confirmed across both pandas and SQL approaches
4. **Priority accounts consistent at $1.2-1.4M avg ARR:** K-means Tier 1 threshold holds globally — Priority flag reliably identifies highest value whitespace regardless of subregion
5. **ANZ and North Asia 45% whitespace:** Structural capacity gap — rep headcount cannot cover account universe without additional hires